In [1]:
# 変化量の上位95%だけを抽出
import pandas as pd
import os

# 入力ファイル
input_csv = "/home/mitani/CSJ-emo-int_bunseki/0718/R_bunseki/vad_dis.csv"

# 出力ファイル
output_csv = "/home/mitani/CSJ-emo-int_bunseki/0718/R_bunseki/results/Δa-95.csv"

# 出力先が無ければ新規作成
os.makedirs(os.path.dirname(output_csv), exist_ok=True)

# CSV読み込み
df = pd.read_csv(input_csv)

filter_95 = df[df["delta_arousal"] > 0.52]

result = filter_95[
    [
        "filename",
        "text",
        "valence",
        "arousal",
        "dominance",
        "delta_valence",
        "delta_arousal",
        "delta_dominance",
    ]
]

# 保存
result.to_csv(output_csv, index=False, encoding="utf-8-sig")

print(f"保存完了: {output_csv}")
print(f"発話数: {len(result)}")

保存完了: /home/mitani/CSJ-emo-int_bunseki/0718/R_bunseki/results/Δa-95.csv
発話数: 12


In [2]:
df[df["delta_arousal"] > 0.52]

,filename,text,valence,arousal,dominance,delta_valence,delta_arousal,delta_dominance
104,0215_R.wav,(F うん)(F うん),0.29,0.00,0.07,0.63,0.78,0.68
105,0219_R.wav,(F はい)そうです,0.60,0.56,0.54,0.31,0.56,0.47
124,0259_R.wav,(F うーん),0.26,0.05,0.11,0.62,0.70,0.60
130,0269_R.wav,んでまたお迎えが早いんですよ,0.65,0.55,0.48,0.34,0.53,0.37
152,0319_R.wav,(F うーん),0.31,0.08,0.15,0.58,0.67,0.57
158,0331_R.wav,<笑>,0.86,0.71,0.69,0.55,0.62,0.59
198,0402_R.wav,<笑>,0.92,0.78,0.76,0.58,0.65,0.57
199,0404_R.wav,そうなんですよ(D お),0.30,0.10,0.21,0.62,0.68,0.55
207,0421_R.wav,<笑>,0.92,0.82,0.77,0.58,0.69,0.59
222,0456_R.wav,要するに何か,0.20,0.65,0.63,0.06,0.58,0.48


In [ ]:
# パーセンタイル95%のRの発話
import pandas as pd
import os
import shutil

# ==========================
# パス
# ==========================
l_csv = "/home/mitani/CSJ-emo-int_bunseki/0718/R_bunseki/vad_dis.csv"
wav_dir = "/home/mitani/CSJ-emo-int_bunseki/ALL-WAV"
output_root = "/home/mitani/CSJ-emo-int_bunseki/0718/R_bunseki/results/eval-a"

os.makedirs(output_root, exist_ok=True)

# ==========================
# CSV読込（Lのみ）
# ==========================
df_l = pd.read_csv(l_csv)

# 発話番号（CSVのインデックスではなく、0015_R.wav の「15」を指定）
change_idx = [
    215, 219, 259, 269, 319, 331, 402, 404, 421, 456, 459, 483
]

context = 3

# ==========================
# 抽出
# ==========================
for utt_no in change_idx:

    # 発話番号からファイル名を作成
    target_filename = f"{utt_no:04d}_R.wav"

    # filenameから該当行を検索
    matched = df_l[df_l["filename"] == target_filename]

    if matched.empty:
        print(f"{target_filename} がCSV内に見つかりません")
        continue

    # DataFrame内でのインデックス取得
    idx = matched.index[0]

    print(f"発話番号 {utt_no} -> DataFrame index {idx}")

    # 前後3発話（Rのみ）
    start = max(0, idx - context)
    end = min(len(df_l), idx + context + 1)

    context_df = df_l.iloc[start:end]

    # 保存フォルダ
    folder = os.path.join(
        output_root,
        os.path.splitext(target_filename)[0]
    )
    os.makedirs(folder, exist_ok=True)

    # context.csv保存
    context_df.to_csv(
        os.path.join(folder, "context.csv"),
        index=False,
        encoding="utf-8-sig"
    )

    # target情報保存
    with open(os.path.join(folder, "target.txt"), "w", encoding="utf-8") as f:
        f.write(f"Utterance number : {utt_no}\n")
        f.write(f"DataFrame index : {idx}\n")
        f.write(f"Target filename : {target_filename}\n")
        f.write(f"Context range : {start} - {end - 1}\n")

    # 音声コピー
    for _, row in context_df.iterrows():

        src = os.path.join(wav_dir, row["filename"])

        if not os.path.exists(src):
            print(f"Not found: {src}")
            continue

        # ターゲット音声にはTARGET_を付ける
        if row["filename"] == target_filename:
            dst_name = "TARGET_" + row["filename"]
        else:
            dst_name = row["filename"]

        dst = os.path.join(folder, dst_name)

        shutil.copy2(src, dst)

print("完了")

発話番号 215 -> DataFrame index 104
発話番号 219 -> DataFrame index 105
発話番号 259 -> DataFrame index 124
発話番号 269 -> DataFrame index 130
発話番号 319 -> DataFrame index 152
発話番号 331 -> DataFrame index 158
発話番号 402 -> DataFrame index 198
発話番号 404 -> DataFrame index 199
発話番号 421 -> DataFrame index 207
発話番号 456 -> DataFrame index 222
発話番号 459 -> DataFrame index 225
発話番号 483 -> DataFrame index 238
完了
